In [ ]:
%%sql -r dataframe_1
CREATE DATABASE IF NOT EXISTS mempool_DB;
USE mempool_DB;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE STORAGE INTEGRATION azure_adls_snowflake_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  ENABLED = TRUE
  AZURE_TENANT_ID = '*****************'
  STORAGE_ALLOWED_LOCATIONS = ('azure://mempoolspaceacc.blob.core.windows.net/bronze/Mempool_parsing');

DESC STORAGE INTEGRATION azure_adls_snowflake_int;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE FILE FORMAT parquet_format
  TYPE = PARQUET;

CREATE OR REPLACE STAGE adls_api_stage
  STORAGE_INTEGRATION = azure_adls_snowflake_int
  URL = 'azure://mempoolspaceacc.blob.core.windows.net/bronze/Mempool_parsing'
  FILE_FORMAT = parquet_format;

LIST @adls_api_stage;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE STG_MEMPOOL_PARSED AS
SELECT
    $1:sequence_number::BIGINT AS sequence_number,
    TO_TIMESTAMP($1:enqueued_time::STRING, 'MM/DD/YYYY HH12:MI:SS AM') AS enqueued_time,
    $1:block_size::BIGINT AS block_size,
    $1:block_vsize::BIGINT AS block_vsize,
    $1:median_fee::DOUBLE AS median_fee,
    $1:n_tx::INT AS n_tx,
    $1:total_fees::BIGINT AS total_fees,
    $1:fee_p10::DOUBLE AS fee_p10,
    $1:fee_p25::DOUBLE AS fee_p25,
    $1:fee_p50::DOUBLE AS fee_p50,
    $1:fee_p75::DOUBLE AS fee_p75,
    $1:fee_p90::DOUBLE AS fee_p90,
    $1:fee_p95::DOUBLE AS fee_p95,
    $1:fee_p99::DOUBLE AS fee_p99
FROM @adls_api_stage
(pattern => '.*\\.parquet');

SELECT * FROM STG_MEMPOOL_PARSED LIMIT 10;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TASK load_mempool_azure_task
  WAREHOUSE = 'MEMPOOL'
  SCHEDULE = '360 MINUTE'
AS
  INSERT INTO STG_MEMPOOL_PARSED
  SELECT
      $1:sequence_number::BIGINT AS sequence_number,
      TO_TIMESTAMP($1:enqueued_time::STRING, 'MM/DD/YYYY HH12:MI:SS AM') AS enqueued_time,
      $1:block_size::BIGINT AS block_size,
      $1:block_vsize::BIGINT AS block_vsize,
      $1:median_fee::DOUBLE AS median_fee,
      $1:n_tx::INT AS n_tx,
      $1:total_fees::BIGINT AS total_fees,
      $1:fee_p10::DOUBLE AS fee_p10,
      $1:fee_p25::DOUBLE AS fee_p25,
      $1:fee_p50::DOUBLE AS fee_p50,
      $1:fee_p75::DOUBLE AS fee_p75,
      $1:fee_p90::DOUBLE AS fee_p90,
      $1:fee_p95::DOUBLE AS fee_p95,
      $1:fee_p99::DOUBLE AS fee_p99
  FROM @adls_api_stage
  (pattern => '.*\\.parquet');

ALTER TASK load_mempool_azure_task RESUME;

In [ ]:
%%sql -r dataframe_4
select count(*) as total from STG_MEMPOOL_PARSED

In [ ]:
%%sql -r dataframe_5
EXECUTE TASK load_mempool_azure_task;